En este cuaderno vamos a usar un modelo ViT (Vision Transformers). El elegido es: *google/vit-base-patch16-224-in21k*

In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset, Image, Features, Value
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import evaluate

# 1. Defiendo las rutas y el modelo con el que vamos a trabajar

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
CSV_IMAGENES = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/dataset_imagenes.csv"
OUTPUT_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/ViT_FineTuned"
MODEL_CHECKPOINT = "google/vit-base-patch16-224-in21k"

In [10]:
print("Cargando el dataset de imágenes...")
df_imagenes = pd.read_csv(CSV_IMAGENES)
df_imagenes

Cargando el dataset de imágenes...


,id_EXIST,path_imagen,label
0,120001,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1
1,120001,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1
2,120001,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1
3,120001,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1
4,120002,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1
...,...,...,...
10025,121523,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,0
10026,121524,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,0
10027,121524,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,0
10028,121524,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,0


# 2. Particionado

**OJO** con este paso, tendremos que tener en cuenta el contenido de la columna *id_EXIST* porque estaríamos falseando los resultados si de un mismo vídeo tenemos un frame en el conjunto de entrenamiento, otro frame en el conjunto de test y otro frame en el conjunto de validación.

In [8]:
# Usamos id_EXIST como 'grupo' para evitar que imágenes del mismo vídeo se separen
gss_train_test = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

# Separamos Train (70%) y Temp (30%)
train_idx, temp_idx = next(gss_train_test.split(df_imagenes, groups=df_imagenes['id_EXIST']))
train_df = df_imagenes.iloc[train_idx]
temp_df = df_imagenes.iloc[temp_idx]

# Ahora separamos el Temp en Validation (15%) y Test (15%)
gss_val_test = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val_test.split(temp_df, groups=temp_df['id_EXIST']))
val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

print(f"Vídeos/Imágenes en Train: {train_df['id_EXIST'].nunique()} / {len(train_df)}")
print(f"Vídeos/Imágenes en Valid: {val_df['id_EXIST'].nunique()} / {len(val_df)}")
print(f"Vídeos/Imágenes en Test: {test_df['id_EXIST'].nunique()} / {len(test_df)}")

Vídeos/Imágenes en Train: 1755 / 7020
Vídeos/Imágenes en Valid: 376 / 1504
Vídeos/Imágenes en Test: 377 / 1506


# 3. Conversión a Formato `Dataset` de Hugging Face

In [11]:
def crear_dataset(dataframe):
    # Primero creamos el dataset leyendo las rutas como texto plano desde el dataframe
    dataset = Dataset.from_pandas(dataframe)

    # Casteamos la columna de texto a tipo Image() para que lea los archivos de Drive
    dataset = dataset.cast_column("path_imagen", Image())

    # Renombramos para que el modelo lo entienda
    return dataset.rename_column("path_imagen", "image")

print("Transformando rutas en imágenes procesables...")
train_dataset = crear_dataset(train_df)
valid_dataset = crear_dataset(val_df)

Transformando rutas en imágenes procesables...


# 4. Procesador de Imágenes (`ViTImageProcessor`)

Esto es igual que el tokenizador de texto, pero para imágenes (redimensiona a 224x224, normaliza colores, etc.)

In [12]:
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

def process_images(batch):
    # Toma una lista de imágenes y las convierte en los tensores matemáticos que ViT necesita
    inputs = processor([img.convert("RGB") for img in batch["image"]], return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

# Aplicamos la transformación "al vuelo" para no saturar la RAM
train_dataset.set_transform(process_images)
valid_dataset.set_transform(process_images)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

# 5. Inicialización del Modelo Base ViT

In [13]:
id2label = {0: "No misógino", 1: "Misógino"}
label2id = {"No misógino": 0, "Misógino": 1}

model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # Necesario porque el modelo in21k original no tiene capa de clasificación
)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.attention.attention.key.bias     | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.weight | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weight           | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.weight          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.weight   | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.bias            | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.bias   | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.bias   | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.b

# 6. Métrica de Evaluación

In [14]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

# Usamos tu amado F1 Score macro
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

# 7. Hiperparámetros del Entrenamiento

In [16]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # IMPORTANTE: En visión debe ser False
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,          # ViT prefiere Learning Rates más bajos que los LLMs
    per_device_train_batch_size=16, # ViT es ligero, aguanta un batch de 16 en Colab
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,                   # Precisión mixta para acelerar en GPU
    report_to="none"
)

# 8. Entrenamiento

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Iniciando entrenamiento visual con ViT...")
trainer.train()

print("💾 Guardando modelo visual...")
trainer.save_model(OUTPUT_DIR + "/modelo_final")
processor.save_pretrained(OUTPUT_DIR + "/modelo_final")
print("✅ ¡Entrenamiento completado!")

🚀 Iniciando entrenamiento visual con ViT...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.690323,0.686397,0.437733,0.539229
2,0.692156,0.688402,0.504317,0.545213


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]